# TRIDENT T69 - custom residual dense 4x TWO-EPOCH pilot

This pilot alone is approved. Architecture: 473,059 parameters, nearest decoder, unchanged approved loss. No longer training, official test evaluation, or inference export is authorized. Execution controls default to false. See docs/custom_pilot/launch.md.

Code from a new reviewed source archive/Git commit; existing dataset and baseline epoch-2 checkpoint stay on Drive. Training runs in the existing Colab CUDA Python environment. No virtual environment or CPU Torch installation.

## 1. Settings and runtime
Use the newly packaged source archive or an exact published Git commit. Keep RUN_ID unchanged when reconnecting. The two-epoch custom pilot is already approved; controls default to false to prevent accidental execution during setup. Enable TRAINING_APPROVED only when ready to launch this pilot.

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, zipfile
SOURCE_MODE = "archive"  # or "github" with an exact reviewed commit
GIT_URL = "https://github.com/Dhanush1756/TRIDENT-Depth-T69.git"
GIT_COMMIT = ""  # exact full 40-character reviewed commit; do not use a moving branch
SOURCE_SHA256 = ""  # required only for source archive mode
DATASET_SHA256 = "0e97b6d1a2ae7a83c4383566eee782dc04dc68048f9aff9590ebe0c7d9df1cd6"  # printed by scripts/pack_colab.py on Windows
DRIVE_ROOT = Path("/content/drive/MyDrive/TRIDENT-T69")
REPO = Path("/content/TRIDENT-Depth-T69")
CACHE = Path("/content/trident-nyu-converted")
RUN_ID = "pilot_dense4_residual_seed69"
RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID
RESUME = ""  # e.g. "epoch_001.pt"; latest complete checkpoint only
TRAINING_APPROVED = False  # approval exists; enable manually to launch only this pilot
RUN_COMPARISON = False  # enable for the paired comparison after epoch two
EXPORT_APPROVED = False
assert (3, 10) <= sys.version_info[:2] < (3, 14), "Use a Python 3.10â€“3.13 runtime"
print(platform.python_version(), platform.platform())
BASELINE = DRIVE_ROOT / "runs/pilot_dense4_lightweight_seed69/epoch_002.pt"
BASELINE_SHA256 = "a1fb5080360d9e693d43e67b3aa105da05be93af6baca6d1c84bee979ce3fb43"


## 2. Mount persistent Drive
Upload `nyu_converted.zip` and optionally `trident_custom_pilot_source.zip` under DRIVE_ROOT. Archive roots must contain the converted files / repository files directly, without an extra folder. Never store credentials in the notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert DRIVE_ROOT.is_dir(), f"Create {DRIVE_ROOT} and upload the archives first"


## 3. Obtain reviewed source
The current local work is not yet on GitHub. Publish the reviewed changes yourself before using GitHub mode, or package them with the documented source command. Existing checkout directories are not overwritten.

In [ ]:
if SOURCE_MODE == "github":
    assert len(GIT_COMMIT) == 40 and all(c in "0123456789abcdefABCDEF" for c in GIT_COMMIT), "Set the reviewed full Git commit"
    if not REPO.exists():
        subprocess.run(["git", "clone", GIT_URL, str(REPO)], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", GIT_COMMIT], check=True)
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    assert actual.lower() == GIT_COMMIT.lower()
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Checkout has local modifications"
    SOURCE_ID = actual
elif SOURCE_MODE == "archive":
    archive = DRIVE_ROOT / "trident_source.zip"
    assert len(SOURCE_SHA256) == 64, "Set source archive SHA256"
    digest = hashlib.sha256()
    with archive.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    assert digest.hexdigest() == SOURCE_SHA256
    assert not REPO.exists(), "Use a fresh runtime or GitHub mode for an existing checkout"
    with zipfile.ZipFile(archive) as bundle:
        names = set()
        for item in bundle.infolist():
            name = item.filename
            target = (REPO / name).resolve()
            assert "\\" not in name and target.is_relative_to(REPO.resolve())
            assert name not in names and (item.external_attr >> 16) & 0o170000 != 0o120000
            names.add(name)
        bundle.extractall(REPO)
    SOURCE_ID = SOURCE_SHA256
else:
    raise ValueError("SOURCE_MODE must be github or archive")
assert (REPO / "trident_depth/training/colab.py").is_file(), "Reviewed Colab changes are missing"
os.chdir(REPO)
BASELINE = DRIVE_ROOT / "runs/pilot_dense4_lightweight_seed69/epoch_002.pt"
BASELINE_SHA256 = "a1fb5080360d9e693d43e67b3aa105da05be93af6baca6d1c84bee979ce3fb43"


## 4. Existing runtime, dependencies and GPU detection
Use the existing Colab Python runtime directly, preserving its CUDA-enabled PyTorch. Only additional requirements are installed; any dependency plan touching PyTorch or its CUDA stack is rejected. The repository installs with dependency resolution disabled. Fresh subprocesses use the updated packages without creating a virtual environment. Record exact Python, torch, CUDA, package versions and GPU, and run the complete project test suite after installation. Resume requires the same software fingerprint; hardware changes may still affect numerical reproducibility.

In [ ]:
# Use Colab's existing runtime directly: no venv or ensurepip.
import tempfile
PY = sys.executable
print("Colab Python:", PY)
torch_probe = """
import json, torch
print(json.dumps(dict(version=str(torch.__version__), cuda=torch.version.cuda,
 location=torch.__file__, cuda_available=torch.cuda.is_available())))
"""
TORCH_BEFORE = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_BEFORE["cuda_available"], "Select a GPU runtime; Colab CUDA PyTorch must already be available"

# Resolve only the additional requirements first. Reject any plan that would
# touch PyTorch or its CUDA stack, then install exactly that plan without resolving
# further dependencies. This cannot pull in a CPU-only torch wheel.
with tempfile.TemporaryDirectory(prefix="trident-pip-") as temporary:
    report = Path(temporary) / "install-plan.json"
    subprocess.run([str(PY), "-m", "pip", "install", "--dry-run",
                    "--only-binary=:all:", "--report", str(report),
                    "-r", str(REPO / "requirements-colab.txt")], check=True)
    plan = json.loads(report.read_text())["install"]
    protected = {"torch", "torchvision", "torchaudio", "torchtext", "triton", "pytorch-triton"}
    for item in plan:
        name = item["metadata"]["name"].lower().replace("_", "-")
        assert name not in protected and not name.startswith("nvidia-"), f"Refusing to replace Colab GPU package: {name}"
    urls = [item["download_info"]["url"] for item in plan]
    if urls:
        subprocess.run([str(PY), "-m", "pip", "install", "--no-deps",
                        "--only-binary=:all:", *urls], check=True)
subprocess.run([str(PY), "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(REPO)], check=True)
TORCH_AFTER = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_AFTER == TORCH_BEFORE, "Colab PyTorch/CUDA changed during setup"
probe = """
import json, platform, torch, numpy, scipy, cv2, PIL, h5py
assert (2, 6) <= tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (3, 0)
print(json.dumps(dict(python=platform.python_version(), torch=str(torch.__version__),
 cuda=torch.version.cuda, numpy=numpy.__version__, scipy=scipy.__version__,
 opencv=cv2.__version__, pillow=PIL.__version__, h5py=h5py.__version__,
 gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
 gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None)))
"""
ENVIRONMENT = json.loads(subprocess.check_output([str(PY), "-c", probe], text=True))
print(json.dumps(ENVIRONMENT, indent=2))
assert ENVIRONMENT["gpu"], "No CUDA GPU available: select a GPU runtime. Do not run the pilot on CPU."
# Complete project suite after installation; CPU tests do not train a model.
subprocess.run([str(PY), "-m", "pytest", "-q"], check=True,
               env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))


## 5. Stage and audit the official dataset
Copy/extract from the uploaded ZIP into temporary Colab storage once per runtime. Archive SHA256 and the approved manifest SHA256 are checked. Preserve 672 training / 123 validation / 654 locked test records and their scene assignments. Test pixels are not loaded by training or validation.

In [ ]:
assert len(DATASET_SHA256) == 64, "Set the dataset archive SHA256 printed on Windows"
stage = """
import sys, json
from pathlib import Path
from trident_depth.training.colab import stage_archive, audit_pilot_dataset
archive, cache, expected = sys.argv[1:]
if not Path(cache).exists():
    stage_archive(archive, cache, expected)
assert (Path(cache) / '.archive_sha256').read_text() == expected, 'Cache belongs to another archive'
print(json.dumps(audit_pilot_dataset(Path(cache) / 'manifest_development.json'), indent=2))
"""
subprocess.run([str(PY), "-c", stage, str(DRIVE_ROOT / "nyu_converted.zip"),
                str(CACHE), DATASET_SHA256], check=True)
MANIFEST = CACHE / "manifest_development.json"


## 6. Verify the approved custom configuration and GPU preflight
No optimizer steps. The preflight checks 473,059 parameters, finite full-frame loss/gradients, active uncertainty gradients, unchanged weights and complete checkpoint/RNG roundtrip in a temporary directory. It verifies the original scene-safe split. Boundary confidence has no direct supervision under the approved loss; boundary F1 is measured on predicted depth edges.

In [ ]:
EXPERIMENT = REPO / "configs/phase5/pilot_dense4_residual.json"
assert EXPORT_APPROVED is False, "Inference export is not approved"
assert BASELINE.is_file(), f"Restore the audited baseline checkpoint to {BASELINE}"
digest = hashlib.sha256(BASELINE.read_bytes()).hexdigest()
assert digest == BASELINE_SHA256, "Baseline checkpoint differs from the audited epoch two"
cfg = json.loads(EXPERIMENT.read_text())
pipeline_file = EXPERIMENT.parent / cfg["pipeline_config"]
pipeline = json.loads(pipeline_file.read_text())
assert (cfg["model"], cfg["epochs"], cfg["batch_size"], cfg["learning_rate"], cfg["seed"], cfg["decoder_upsampling"]) == ("residual_cnn", 2, 1, 1e-4, 69, "nearest")
assert (pipeline["protocol"], pipeline["scale"], pipeline["target_size"]) == ("dense_sr", 4, None)
preflight_command = [str(PY), "-m", "trident_depth.training.custom_pilot",
                     "--experiment", str(EXPERIMENT), "--manifest", str(MANIFEST), "--device", "cuda"]
PREFLIGHT = json.loads(subprocess.check_output(preflight_command, text=True))
assert PREFLIGHT["parameters"] == 473059 and PREFLIGHT["optimizer_steps"] == 0 and PREFLIGHT["cuda_verified"]
print(json.dumps(PREFLIGHT, indent=2))
REFERENCE_FRAMES = REPO / "docs/pilot_dense4/per_frame.json"
assert REFERENCE_FRAMES.is_file(), "Source archive must include the audited 123-frame reference"
RECORD = {"source_id": SOURCE_ID, "environment": ENVIRONMENT,
          "dataset_archive_sha256": DATASET_SHA256,
          "experiment_sha256": hashlib.sha256(EXPERIMENT.read_bytes()).hexdigest(),
          "pipeline_sha256": hashlib.sha256(pipeline_file.read_bytes()).hexdigest(),
          "baseline_checkpoint_sha256": BASELINE_SHA256,
          "reference_frames_sha256": hashlib.sha256(REFERENCE_FRAMES.read_bytes()).hexdigest(),
          "boundary_evaluation": PREFLIGHT["boundary_evaluation"]}
ENV_FILE = RUN_DIR.parent / (RUN_ID + "_environment.json")
if ENV_FILE.exists():
    assert json.loads(ENV_FILE.read_text()) == RECORD, "Resume provenance differs; restore original source/environment/data"
if RESUME:
    assert RESUME == "epoch_001.pt", "Only epoch one can resume within this two-epoch budget"
    assert (RUN_DIR / RESUME).is_file(), "Selected complete checkpoint is missing"
else:
    assert not RUN_DIR.exists(), "Existing run: resume epoch one, or compare the completed pilot"


## 7. Launch or resume the approved TWO-EPOCH pilot
Set TRAINING_APPROVED=True to execute the already approved pilot. The launch uses only pilot_dense4_residual.json, records boundary F1 after every epoch and stops after epoch two. Reconnect with identical source/environment/RUN_ID and RESUME="epoch_001.pt" to run epoch two only. After epoch two, run comparison without rerunning training. No inference export is included.

In [ ]:
if not TRAINING_APPROVED:
    print("Training disabled. No optimizer steps performed.")
else:
    assert ENVIRONMENT["gpu"] and MANIFEST.is_file()
    RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not ENV_FILE.exists():
        ENV_FILE.write_text(json.dumps(RECORD, indent=2))
        freeze = subprocess.check_output([str(PY), "-m", "pip", "freeze"], text=True)
        ENV_FILE.with_suffix(".freeze.txt").write_text(freeze)
    command = [str(PY), "-m", "trident_depth.training.train",
               "--experiment", str(EXPERIMENT), "--manifest", str(MANIFEST),
               "--output", str(RUN_DIR), "--device", "cuda", "--record-boundary-f1", "--provenance", str(ENV_FILE), "--execute-training"]
    if RESUME:
        command += ["--resume", str(RUN_DIR / RESUME)]
    subprocess.run(command, check=True)


## 8. Paired validation comparison only - no weight export
Set RUN_COMPARISON=True after both epochs finish. Compare bicubic, the audited baseline epoch two, and the custom checkpoint selected by validation RMSE. All methods use the original 123 IDs, masks and data-file hashes. Global RMSE/MAE and frame-mean boundary F1 are recomputed together. Results go to a new comparison folder inside the custom run. Existing benchmark records stay unchanged. Stop here; no additional training or export follows.

In [ ]:
assert EXPORT_APPROVED is False, "Inference export is not approved"
if not RUN_COMPARISON:
    print("Comparison deferred. Enable after both custom epochs complete.")
else:
    comparison_command = [str(PY), "-m", "trident_depth.evaluation.custom_pilot",
                          "--custom-run", str(RUN_DIR), "--baseline-checkpoint", str(BASELINE),
                          "--manifest", str(MANIFEST), "--reference-frames", str(REFERENCE_FRAMES),
                          "--output", str(RUN_DIR / "validation_comparison"), "--device", "cuda"]
    subprocess.run(comparison_command, check=True)
    print("Download the custom run folder for audit. No inference weights were exported.")
